In [22]:
# File and directory operations
import os

# Numerical computations and array handling
import numpy as np

# Data manipulation and CSV handling
import pandas as pd

# Plotting and visualization
import matplotlib.pyplot as plt

# Deep learning framework for model building and training
import tensorflow as tf

# Rich display utilities for Jupyter notebooks (e.g., HTML tables, styled output)
from IPython.display import HTML, display

In [23]:


# Path to the CSV file containing bounding box annotations
csv_path = 'C:/Users/samya/PyCharmProject/Pneumonia-Detection_dataset/data/stage_2_train_labels.csv'

# Load the full dataset
labels_df = pd.read_csv(csv_path)

# Filter to include only rows where pneumonia is present (Target == 1)
pneumonia_df = labels_df[labels_df['Target'] == 1].copy()

#reset index for cleaner downstream processing
pneumonia_df.reset_index(drop=True, inplace=True)

In [24]:

def plot_score(hist):
    fig, ax = plt.subplots(5, 1, figsize=(10, 20))  # Corrected 'subplot' to 'subplots'
    ax = ax.ravel()

    for i, met in enumerate(['accuracy', 'precision', 'recall', 'AUC', 'loss']):
        ax[i].plot(hist.history[met])
        ax[i].plot(hist.history['val_' + met])
        ax[i].set_title(f'Model {met}')
        ax[i].set_xlabel('Epochs')
        ax[i].set_ylabel(met)
        ax[i].legend(['Train', 'Validation'])

    plt.tight_layout()
    plt.show()


In [47]:


# Set directory where .npy files are stored
npy_dir = 'npy_data'
test_Y = np.load(os.path.join(npy_dir, 'test_masks.npy'))  # or y_mask if loaded earlier
test_Y = test_Y / 255.0 if test_Y.max() > 1 else test_Y     # Normalize to [0, 1]
test_Y = np.expand_dims(test_Y, axis=-1) if test_Y.ndim == 3 else test_Y  # Add channel dim
test_Y = test_Y.astype('float32')                           # Match model expectations
'''# Load preprocessed datasets
train_X_rgb = np.load(os.path.join(npy_dir, 'train_X_rgb.npy'))
train_Y     = np.load(os.path.join(npy_dir, 'train_Y.npy'))
test_X_rgb  = np.load(os.path.join(npy_dir, 'test_X_rgb.npy'))
test_Y      = np.load(os.path.join(npy_dir, 'test_Y.npy'))
y_mask      = np.load(os.path.join(npy_dir, 'train_masks.npy')) '''

FileNotFoundError: [Errno 2] No such file or directory: 'npy_data\\test_masks.npy'

In [26]:
y_mask

array([[[[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        ...,

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]]],


       [[[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        ...,

        [[0],
         [0],
         [0],
         ...,
         [0],


In [27]:
print("Total non-zero pixels in masks:", np.count_nonzero(y_mask))

Total non-zero pixels in masks: 2892187


In [28]:


labels = pd.read_csv(r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_train_labels.csv")
count_normal = (labels['Target'] == 0).sum()
count_pneumonia = (labels['Target'] == 1).sum()
train_count = len(train_X_rgb)

classweight = {
    0: (1 / count_normal) * (train_count / 2.0),
    1: (1 / count_pneumonia) * (train_count / 2.0)
}

In [29]:
from tensorflow import keras
from tensorflow.keras.layers import *
from tensorflow.keras import Model


In [30]:
METRICS = [
    'accuracy',
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='AUC'),
    tf.keras.metrics.TruePositives(name='tp'),
    tf.keras.metrics.TrueNegatives(name='tn'),
    tf.keras.metrics.FalsePositives(name='fp'),
    tf.keras.metrics.FalseNegatives(name='fn'),
    tf.keras.metrics.SpecificityAtSensitivity(0.9, name='specificity_at_sens_90'),
    tf.keras.metrics.SensitivityAtSpecificity(0.9, name='sensitivity_at_spec_90'),
]

In [31]:
import tensorflow as tf

# Flexible Exponential Decay Function
def get_exponential_decay_fn(lr_initial=0.01, decay_steps=20, decay_rate=0.1):
    """
    Returns a learning rate function using exponential decay.
    """
    return lambda epoch: lr_initial * decay_rate ** (epoch / decay_steps)

# Returns a LearningRateScheduler callback with decay configuration
def get_lr_scheduler_cb(lr_initial=0.01, decay_steps=20, decay_rate=0.1):
    return tf.keras.callbacks.LearningRateScheduler(
        schedule=get_exponential_decay_fn(lr_initial, decay_steps, decay_rate),
        verbose=1
    )

# Returns a ModelCheckpoint callback that saves to a unique file
def get_model_checkpoint_cb(model_name):
    return tf.keras.callbacks.ModelCheckpoint(
        filepath=f"{model_name}.h5",
        save_best_only=True,
        monitor="val_loss",
        mode="min",
        verbose=1
    )

# Shared EarlyStopping callback
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    patience=5,
    restore_best_weights=True,
    monitor="val_loss",
    mode="min",
    verbose=1
)

In [32]:
checkpoint_cb = get_model_checkpoint_cb("mn_cnn_model")
lr_scheduler_cb = get_lr_scheduler_cb()

In [33]:
import numpy as np

def create_segmentation_mask(img_id, boxes_df, orig_size=(1024, 1024), new_size=(64, 64)):
    """
    Generate a binary segmentation mask from bounding boxes for a given image ID.

    Parameters:
        img_id (str or int): Identifier for the image.
        boxes_df (pd.DataFrame): DataFrame containing bounding box info with columns:
                                 ['patientId', 'x', 'y', 'width', 'height'].
        orig_size (tuple): Original image size (width, height).
        new_size (tuple): Desired output mask size (width, height).

    Returns:
        np.ndarray: Binary mask of shape `new_size` with 1s inside bounding boxes.
    """
    mask = np.zeros(new_size, dtype=np.uint8)

    # Filter boxes for the given image ID
    boxes = boxes_df[boxes_df['patientId'] == img_id]

    scale_x = new_size[0] / orig_size[0]
    scale_y = new_size[1] / orig_size[1]

    for _, row in boxes.iterrows():
        x1 = int(row['x'] * scale_x)
        y1 = int(row['y'] * scale_y)
        x2 = int((row['x'] + row['width']) * scale_x)
        y2 = int((row['y'] + row['height']) * scale_y)

        # Clip coordinates to mask bounds
        x1, x2 = np.clip([x1, x2], 0, new_size[0])
        y1, y2 = np.clip([y1, y2], 0, new_size[1])

        mask[y1:y2, x1:x2] = 1

    return mask

In [35]:
import numpy as np
import os
from sklearn.model_selection import train_test_split

# --- Configuration ---
npy_dir = 'npy_data'           # Directory containing .npy files
subset_size = 7000             # Limit dataset size (for faster experimentation)


# --- Reduce Dataset (Optional) ---
X = train_X_rgb[:subset_size]
y = y_mask[:subset_size]

# --- Split into Train/Validation ---
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# --- Inspect Result ---
print("Training set shape:   ", X_train.shape, y_train.shape)
print("Validation set shape: ", X_val.shape, y_val.shape)

Training set shape:    (5600, 64, 64, 3) (5600, 64, 64, 1)
Validation set shape:  (1400, 64, 64, 3) (1400, 64, 64, 1)


In [ ]:
from tensorflow.keras import layers, models, Input

def build_unet(input_shape=(64, 64, 3)):
    inputs = Input(input_shape)

    # --- Encoder ---
    def conv_block(x, filters):
        x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
        x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
        return x

    def encoder_block(x, filters):
        f = conv_block(x, filters)
        p = layers.MaxPooling2D(pool_size=(2, 2))(f)
        return f, p

    def decoder_block(x, skip, filters):
        x = layers.Conv2DTranspose(filters, kernel_size=2, strides=2, padding='same')(x)
        x = layers.Concatenate()([x, skip])
        return conv_block(x, filters)

    # Downsampling path
    f1, p1 = encoder_block(inputs, 32)
    f2, p2 = encoder_block(p1, 64)
    f3, p3 = encoder_block(p2, 128)

    # Bottleneck
    b = conv_block(p3, 256)

    # Upsampling path
    d1 = decoder_block(b, f3, 128)
    d2 = decoder_block(d1, f2, 64)
    d3 = decoder_block(d2, f1, 32)

    # Output layer
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(d3)

    model = models.Model(inputs, outputs, name='UNet')
    return model

In [37]:
unet_model = build_unet((64, 64, 3))  # Assuming your U-Net constructor is ready

unet_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=METRICS
)
unet_model.summary()

Model: "UNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │      9,248 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 32, 32,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │     36,928 │ conv2d_2[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 16, 16,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 16, 16,    │    147,584 │ conv2d_4[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 8, 8, 128) │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 8, 8, 256) │    295,168 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 8, 8, 256) │    590,080 │ conv2d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 16, 16,    │    131,200 │ conv2d_7[0][0]    │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 16,    │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 256)              │            │ conv2d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 16, 16,    │    295,040 │ concatenate[0][0] │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 16, 16,    │    147,584 │ conv2d_8[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_1  │ (None, 32, 32,    │     32,832 │ conv2d_9[0][0]    │
│ (Conv2DTranspose)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,925,601 (7.35 MB)

 Trainable params: 1,925,601 (7.35 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:


history = unet_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    validation_split=0.15,
    class_weight=classweight,
    callbacks=[checkpoint_cb, early_stopping_cb, lr_scheduler_cb],
    verbose=1

)


Epoch 1: LearningRateScheduler setting learning rate to 0.01.
Epoch 1/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - AUC: 0.4806 - accuracy: 0.9454 - fn: 394233.0000 - fp: 92637.5000 - loss: 13.9225 - precision: 0.0399 - recall: 0.0231 - sensitivity_at_spec_90: 0.0771 - specificity_at_sens_90: 0.0361 - tn: 11010837.0000 - tp: 3861.2285
Epoch 1: val_loss improved from inf to 0.14944, saving model to mn_cnn_model.h5


350/350 ━━━━━━━━━━━━━━━━━━━━ 413s 1s/step - AUC: 0.4808 - accuracy: 0.9454 - fn: 395375.2188 - fp: 92650.5625 - loss: 13.8927 - precision: 0.0399 - recall: 0.0231 - sensitivity_at_spec_90: 0.0771 - specificity_at_sens_90: 0.0364 - tn: 11042262.0000 - tp: 3861.8462 - val_AUC: 0.7953 - val_accuracy: 0.9652 - val_fn: 199377.0000 - val_fp: 0.0000e+00 - val_loss: 0.1494 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_sensitivity_at_spec_90: 0.3149 - val_specificity_at_sens_90: 0.5866 - val_tn: 5535023.0000 - val_tp: 0.0000e+00 - learning_rate: 0.0100

Epoch 2: LearningRateScheduler setting learning rate to 0.008912509381337455.
Epoch 2/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - AUC: 0.7227 - accuracy: 0.9653 - fn: 389673.6875 - fp: 26863.6680 - loss: 0.1385 - precision: 0.0147 - recall: 0.0015 - sensitivity_at_spec_90: 0.1986 - specificity_at_sens_90: 0.4193 - tn: 11084160.0000 - tp: 872.4800
Epoch 2: val_loss improved from 0.14944 to 0.14086, saving model to mn_cnn_model.h5


350/350 ━━━━━━━━━━━━━━━━━━━━ 396s 1s/step - AUC: 0.7226 - accuracy: 0.9653 - fn: 390835.2188 - fp: 26950.4707 - loss: 0.1386 - precision: 0.0148 - recall: 0.0015 - sensitivity_at_spec_90: 0.1985 - specificity_at_sens_90: 0.4191 - tn: 11115490.0000 - tp: 875.2991 - val_AUC: 0.7577 - val_accuracy: 0.9652 - val_fn: 199377.0000 - val_fp: 0.0000e+00 - val_loss: 0.1409 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_sensitivity_at_spec_90: 0.2629 - val_specificity_at_sens_90: 0.4163 - val_tn: 5535023.0000 - val_tp: 0.0000e+00 - learning_rate: 0.0089

Epoch 3: LearningRateScheduler setting learning rate to 0.007943282347242816.
Epoch 3/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 881ms/step - AUC: 0.6838 - accuracy: 0.9656 - fn: 403638.1562 - fp: 0.0000e+00 - loss: 0.1417 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.1625 - specificity_at_sens_90: 0.3637 - tn: 11097928.0000 - tp: 0.0000e+00
Epoch 3: val_loss improved from 0.14086 to 0.13930, saving model to mn_cnn_

350/350 ━━━━━━━━━━━━━━━━━━━━ 326s 932ms/step - AUC: 0.6839 - accuracy: 0.9656 - fn: 404765.1875 - fp: 0.0000e+00 - loss: 0.1417 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.1625 - specificity_at_sens_90: 0.3637 - tn: 11129383.0000 - tp: 0.0000e+00 - val_AUC: 0.7588 - val_accuracy: 0.9652 - val_fn: 199377.0000 - val_fp: 0.0000e+00 - val_loss: 0.1393 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_sensitivity_at_spec_90: 0.0674 - val_specificity_at_sens_90: 0.5129 - val_tn: 5535023.0000 - val_tp: 0.0000e+00 - learning_rate: 0.0079

Epoch 4: LearningRateScheduler setting learning rate to 0.0070794578438413795.
Epoch 4/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - AUC: 0.7204 - accuracy: 0.9651 - fn: 401264.9688 - fp: 0.0000e+00 - loss: 0.1397 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.2007 - specificity_at_sens_90: 0.4072 - tn: 11100302.0000 - tp: 0.0000e+00
Epoch 4: val_loss improved from 0.13930 to 0.13654, saving model

350/350 ━━━━━━━━━━━━━━━━━━━━ 430s 1s/step - AUC: 0.7204 - accuracy: 0.9651 - fn: 402398.7812 - fp: 0.0000e+00 - loss: 0.1397 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.2008 - specificity_at_sens_90: 0.4072 - tn: 11131750.0000 - tp: 0.0000e+00 - val_AUC: 0.7657 - val_accuracy: 0.9652 - val_fn: 199377.0000 - val_fp: 0.0000e+00 - val_loss: 0.1365 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_sensitivity_at_spec_90: 0.2342 - val_specificity_at_sens_90: 0.4404 - val_tn: 5535023.0000 - val_tp: 0.0000e+00 - learning_rate: 0.0071

Epoch 5: LearningRateScheduler setting learning rate to 0.006309573444801933.
Epoch 5/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 999ms/step - AUC: 0.7403 - accuracy: 0.9654 - fn: 399549.3125 - fp: 0.0000e+00 - loss: 0.1368 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.2311 - specificity_at_sens_90: 0.4203 - tn: 11102019.0000 - tp: 0.0000e+00
Epoch 5: val_loss improved from 0.13654 to 0.13322, saving model 

...

350/350 ━━━━━━━━━━━━━━━━━━━━ 263s 752ms/step - AUC: 0.8461 - accuracy: 0.9652 - fn: 405885.7812 - fp: 0.0000e+00 - loss: 0.1214 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.4705 - specificity_at_sens_90: 0.6119 - tn: 11128262.0000 - tp: 0.0000e+00 - val_AUC: 0.8494 - val_accuracy: 0.9652 - val_fn: 199377.0000 - val_fp: 0.0000e+00 - val_loss: 0.1209 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_sensitivity_at_spec_90: 0.4867 - val_specificity_at_sens_90: 0.5770 - val_tn: 5535023.0000 - val_tp: 0.0000e+00 - learning_rate: 7.9433e-04

Epoch 24: LearningRateScheduler setting learning rate to 0.0007079457843841381.
Epoch 24/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 770ms/step - AUC: 0.8418 - accuracy: 0.9642 - fn: 408155.8438 - fp: 0.0000e+00 - loss: 0.1249 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.4658 - specificity_at_sens_90: 0.5907 - tn: 11093411.0000 - tp: 0.0000e+00
Epoch 24: val_loss improved from 0.12092 to 0.12062, sa

350/350 ━━━━━━━━━━━━━━━━━━━━ 289s 827ms/step - AUC: 0.8418 - accuracy: 0.9642 - fn: 409270.0312 - fp: 0.0000e+00 - loss: 0.1249 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.4658 - specificity_at_sens_90: 0.5907 - tn: 11124879.0000 - tp: 0.0000e+00 - val_AUC: 0.8501 - val_accuracy: 0.9652 - val_fn: 199377.0000 - val_fp: 0.0000e+00 - val_loss: 0.1206 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_sensitivity_at_spec_90: 0.4890 - val_specificity_at_sens_90: 0.6240 - val_tn: 5535023.0000 - val_tp: 0.0000e+00 - learning_rate: 7.0795e-04

Epoch 25: LearningRateScheduler setting learning rate to 0.0006309573444801933.
Epoch 25/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 735ms/step - AUC: 0.8475 - accuracy: 0.9644 - fn: 407022.4062 - fp: 0.0000e+00 - loss: 0.1230 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.4740 - specificity_at_sens_90: 0.6246 - tn: 11094544.0000 - tp: 0.0000e+00
Epoch 25: val_loss improved from 0.12062 to 0.12043, sa

350/350 ━━━━━━━━━━━━━━━━━━━━ 272s 776ms/step - AUC: 0.8475 - accuracy: 0.9644 - fn: 408139.8125 - fp: 0.0000e+00 - loss: 0.1230 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.4739 - specificity_at_sens_90: 0.6245 - tn: 11126008.0000 - tp: 0.0000e+00 - val_AUC: 0.8502 - val_accuracy: 0.9652 - val_fn: 199377.0000 - val_fp: 0.0000e+00 - val_loss: 0.1204 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_sensitivity_at_spec_90: 0.4973 - val_specificity_at_sens_90: 0.6206 - val_tn: 5535023.0000 - val_tp: 0.0000e+00 - learning_rate: 6.3096e-04

Epoch 26: LearningRateScheduler setting learning rate to 0.0005623413251903491.
Epoch 26/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 836ms/step - AUC: 0.8481 - accuracy: 0.9660 - fn: 388183.9062 - fp: 0.0000e+00 - loss: 0.1188 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.4841 - specificity_at_sens_90: 0.6115 - tn: 11113382.0000 - tp: 0.0000e+00
Epoch 26: val_loss improved from 0.12043 to 0.12035, sa

350/350 ━━━━━━━━━━━━━━━━━━━━ 314s 897ms/step - AUC: 0.8481 - accuracy: 0.9660 - fn: 389354.9688 - fp: 0.0000e+00 - loss: 0.1188 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.4841 - specificity_at_sens_90: 0.6115 - tn: 11144792.0000 - tp: 0.0000e+00 - val_AUC: 0.8504 - val_accuracy: 0.9652 - val_fn: 199377.0000 - val_fp: 0.0000e+00 - val_loss: 0.1203 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_sensitivity_at_spec_90: 0.5035 - val_specificity_at_sens_90: 0.6161 - val_tn: 5535023.0000 - val_tp: 0.0000e+00 - learning_rate: 5.6234e-04

Epoch 27: LearningRateScheduler setting learning rate to 0.0005011872336272723.
Epoch 27/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 699ms/step - AUC: 0.8480 - accuracy: 0.9656 - fn: 392901.0000 - fp: 0.0000e+00 - loss: 0.1200 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.4867 - specificity_at_sens_90: 0.6043 - tn: 11108667.0000 - tp: 0.0000e+00
Epoch 27: val_loss improved from 0.12035 to 0.12005, sa

350/350 ━━━━━━━━━━━━━━━━━━━━ 264s 754ms/step - AUC: 0.8480 - accuracy: 0.9656 - fn: 394058.6250 - fp: 0.0000e+00 - loss: 0.1200 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.4867 - specificity_at_sens_90: 0.6042 - tn: 11140090.0000 - tp: 0.0000e+00 - val_AUC: 0.8526 - val_accuracy: 0.9652 - val_fn: 199377.0000 - val_fp: 0.0000e+00 - val_loss: 0.1200 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_sensitivity_at_spec_90: 0.4959 - val_specificity_at_sens_90: 0.6252 - val_tn: 5535023.0000 - val_tp: 0.0000e+00 - learning_rate: 5.0119e-04

Epoch 28: LearningRateScheduler setting learning rate to 0.00044668359215096305.
Epoch 28/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 824ms/step - AUC: 0.8476 - accuracy: 0.9666 - fn: 390089.4688 - fp: 0.0000e+00 - loss: 0.1175 - precision: 0.0000e+00 - recall: 0.0000e+00 - sensitivity_at_spec_90: 0.4863 - specificity_at_sens_90: 0.6092 - tn: 11111478.0000 - tp: 0.0000e+00
Epoch 28: val_loss did not improve from 0.12005
350/35

In [39]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Evaluate the trained model on the test dataset and return results as a dictionary
results = unet_model.evaluate(test_X_rgb, test_Y, return_dict=True)

# Display evaluation metrics in a readable format
print("\nEvaluation Results:")
for metric, value in results.items():
    print(f"{metric:>20}: {value:.4f}")

# Plot training and validation metrics (assumes 'plot_score' is a custom function)
plot_score(history)

# Predict probabilities on the test set
pred_probs = unet_model.predict(test_X_rgb)

# Convert predicted probabilities to binary class labels (threshold = 0.5)
pred_labels = (pred_probs > 0.5).astype("int32")

# Generate and print the confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(test_Y, pred_labels))

# Print detailed classification report (precision, recall, F1, etc.)
print("\nClassification Report:")
print(classification_report(test_Y, pred_labels, digits=4))

ValueError: Arguments `target` and `output` must have the same rank (ndim). Received: target.shape=(None,), output.shape=(None, 64, 64, 1)

In [45]:
# Clear the previous version entirely
del test_Y

# Reload or rebuild from source
test_Y = np.stack(list_of_masks, axis=0)
test_Y = np.expand_dims(test_Y, axis=-1)
test_Y = test_Y / 255.0
test_Y = test_Y.astype('float32')

print("New test_Y shape:", test_Y.shape)

NameError: name 'list_of_masks' is not defined